# Lab 3 · Models, evaluation, registry and serving
**Block 2 · about 75 min.**
A *run* records how you trained. A **logged model** is the packaged result: weights, preprocessing,
dependencies and a typed **signature**. The **registry** decides which version production uses.

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")          # churn_data.py, churn_model.py
warnings.filterwarnings("ignore")

import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")   # the server you started in the terminal
mlflow.set_experiment("churn-workshop")
print("MLflow", mlflow.__version__)

In [ ]:
import pandas as pd
from mlflow.models import infer_signature
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from churn_data import history, july, time_split, FEATURES, TARGET
from churn_model import make_pipeline, baseline, ChurnDecisionModel, XGB_TRUSTED_TYPES

X_tr, y_tr, X_va, y_va = time_split(history())
BEST = {"max_depth": 2, "n_estimators": 500, "learning_rate": 0.05, "subsample": 0.8}  # or your Lab 2 winner

## Step 1 · Log a model with a signature and an input example
The **signature** is the contract: column names and types in, prediction type out.

In [ ]:
with mlflow.start_run(run_name="xgb_champion_candidate") as run:
    xgb = make_pipeline(XGBClassifier(**BEST, eval_metric="logloss")).fit(X_tr, y_tr)
    proba = xgb.predict_proba(X_va)[:, 1]
    mlflow.log_params(BEST)
    mlflow.log_metric("val_auc", roc_auc_score(y_va, proba))

    xgb_info = mlflow.sklearn.log_model(
        xgb, name="churn_xgb",
        signature=infer_signature(X_va, proba),
        input_example=X_va.head(3),
        skops_trusted_types=XGB_TRUSTED_TYPES,   # safe serialization: trust XGBoost explicitly
    )

print("model_uri:", xgb_info.model_uri)
print("model_id :", xgb_info.model_id)
print(xgb_info.signature)

👉 **In the UI:** experiment → **Models** tab (logged models). Open `churn_xgb`: signature, files, linked run and metrics.

## Step 2 · Load it back as a generic `pyfunc`
Any flavor (sklearn, xgboost, pytorch…) can be loaded through one interface: `predict(DataFrame)`.

In [ ]:
loaded = mlflow.pyfunc.load_model(xgb_info.model_uri)
print(loaded.predict(X_va.head(5)))   # pyfunc calls predict() -> class labels, not probabilities

# The signature is enforced: a missing column is caught before it reaches the model
try:
    loaded.predict(X_va.drop(columns=["TICKETS_30D"]).head(5))
except Exception as e:
    print("Rejected:", type(e).__name__, str(e)[:120])

## Step 3 · A custom pyfunc: ship the business decision with the model
The generic pyfunc returned 0/1 labels. The CRM team needs more than a probability: a risk band and a yes/no for the retention offer.
The threshold becomes a **versioned model parameter** that callers can override per request.

In [ ]:
decision = ChurnDecisionModel(threshold=0.35)
example = X_va[X_va.TICKETS_30D >= 3].head(5)     # a few at-risk customers
with mlflow.start_run(run_name="churn_decision_model"):
    decision_info = mlflow.pyfunc.log_model(
        name="churn_decision",
        python_model=decision,
        artifacts={"classifier": xgb_info.model_uri},
        code_paths=["../src/churn_model.py", "../src/churn_data.py"],
        signature=infer_signature(example, pd.DataFrame({
            "churn_probability": [0.5], "risk_band": ["HIGH"], "send_offer": [True]}),
            params={"threshold": 0.35}),
        input_example=example,
    )

dm = mlflow.pyfunc.load_model(decision_info.model_uri)
print(dm.predict(example))
print(dm.predict(example, params={"threshold": 0.1}))   # a more aggressive campaign

## Step 4 · Evaluate like a reviewer
`mlflow.models.evaluate` computes a standard set of metrics and plots and logs them to the run.

In [ ]:
# pandas 3 uses a new string dtype that evaluate() does not handle yet -> cast to object
valid = X_va.assign(**{TARGET: y_va.values}).astype({"COUNTRY": object})
with mlflow.start_run(run_name="evaluate_xgb"):
    result = mlflow.models.evaluate(xgb_info.model_uri, data=valid, targets=TARGET,
                                    model_type="classifier")
{k: round(v, 4) for k, v in result.metrics.items()
 if k in ("roc_auc", "precision_score", "recall_score", "f1_score", "accuracy_score")}

👉 **Discuss:** accuracy is above 0.90 but recall is low. Why is accuracy misleading when only ~10% of customers churn?
In the UI, open the `evaluate_xgb` run → *Artifacts*: ROC, precision-recall, lift and calibration plots were logged for you.

## Step 5 · Register versions and use aliases
- **Version 1** = logistic baseline, **version 2** = XGBoost.
- **Aliases** are movable pointers: `@champion` is what production loads. Promotion and rollback = moving one alias.

In [ ]:
from mlflow import MlflowClient
client = MlflowClient()
NAME = "churn-model"

with mlflow.start_run(run_name="logreg_for_registry"):
    lr = baseline().fit(X_tr, y_tr)
    mlflow.log_metric("val_auc", roc_auc_score(y_va, lr.predict_proba(X_va)[:, 1]))
    v1 = mlflow.sklearn.log_model(lr, name="churn_logreg", input_example=X_va.head(3),
                                  registered_model_name=NAME)

v2 = mlflow.register_model(xgb_info.model_uri, NAME)      # register an already-logged model
print("versions:", v1.registered_model_version, v2.version)

In [ ]:
client.update_registered_model(NAME, description="Monthly churn score for the CRM retention campaign.")
client.set_model_version_tag(NAME, v2.version, "validation", "passed")
client.set_model_version_tag(NAME, v2.version, "trained_on", "2026-02..2026-04")
client.set_registered_model_alias(NAME, "champion", v2.version)
client.set_registered_model_alias(NAME, "baseline", v1.registered_model_version)

champion = mlflow.pyfunc.load_model(f"models:/{NAME}@champion")
print("champion is version", client.get_model_version_by_alias(NAME, "champion").version)

👉 **In the UI:** **Models** (top bar) → `churn-model`: versions, aliases, tags, description.

## Step 6 · Register the decision model too, then batch-score July

In [ ]:
dv = mlflow.register_model(decision_info.model_uri, "churn-decision")
client.set_registered_model_alias("churn-decision", "champion", dv.version)

jul = july()
scores = mlflow.pyfunc.load_model("models:/churn-decision@champion").predict(jul[FEATURES])
out = pd.concat([jul[["CUSTOMER_ID", "SNAPSHOT_TS"]], scores], axis=1)
out.to_csv("../data/scores_july.csv", index=False)
print(out["risk_band"].value_counts(), "\noffers to send:", int(out["send_offer"].sum()))

## Step 7 · Serve the champion as a REST API
In a **second terminal**, from the workshop folder:
```
export MLFLOW_TRACKING_URI=http://127.0.0.1:5000
mlflow models serve -m "models:/churn-decision@champion" -p 5001 --env-manager local
```
Then run the next cell.

In [ ]:
import requests, time
payload = {"dataframe_split": jul[jul.TICKETS_30D >= 3][FEATURES].head(3).to_dict(orient="split"),
           "params": {"threshold": 0.3}}
payload["dataframe_split"].pop("index", None)
for attempt in range(20):                      # the server takes a few seconds to start
    try:
        r = requests.post("http://127.0.0.1:5001/invocations", json=payload, timeout=30)
        break
    except requests.ConnectionError:
        time.sleep(3)
print(r.status_code, r.json())

## ✍️ Your turn (10 min)
1. Call the API with `curl` from a terminal (copy the JSON from the cell above).
2. Move `@champion` on `churn-model` back to version 1 (a rollback), load `models:/churn-model@champion`
   and check which version you got. Then move it back to version 2.
3. Set a tag `approved_by` on version 2. Where would that tag matter in a real company?